In [ ]:
!pip install flash-attn
!pip install pandasql

In [ ]:
torch.set_default_tensor_type(torch.cuda.FloatTensor)

In [ ]:
#enable flash attention
with torch.backends.cuda.sdp_kernel(
    enable_flash=True,
    enable_math=False,
    enable_mem_efficient=False
):

In [ ]:
!pip install kaggle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
#from flash_attn import flash_attn_func
import kagglehub
import pandas as pd
#from pandasql import sqldf

# Download latest version
#path = kagglehub.dataset_download("harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows")


In [ ]:
#torch.set_default_dtype(torch.cuda.FloatTensor)

In [ ]:
print("Path to dataset files:", )
maintable=kagglehub.load_dataset(kagglehub.KaggleDatasetAdapter.PANDAS,"patmendoza/myanimelist-api","anime_table.csv",pandas_kwargs={"encoding":"ISO-8859-1"})
Genrestableweak=kagglehub.load_dataset(kagglehub.KaggleDatasetAdapter.PANDAS,"patmendoza/myanimelist-api","anime_genres_table.csv",pandas_kwargs={"encoding":"ISO-8859-1"})
genresi=kagglehub.load_dataset(kagglehub.KaggleDatasetAdapter.PANDAS,"patmendoza/myanimelist-api","genres_l.csv",pandas_kwargs={"encoding":"ISO-8859-1"})

Path to dataset files:


/tmp/ipykernel_31/1776351084.py:2: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  maintable=kagglehub.load_dataset(kagglehub.KaggleDatasetAdapter.PANDAS,"patmendoza/myanimelist-api","anime_table.csv",pandas_kwargs={"encoding":"ISO-8859-1"})
/tmp/ipykernel_31/1776351084.py:3: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  Genrestableweak=kagglehub.load_dataset(kagglehub.KaggleDatasetAdapter.PANDAS,"patmendoza/myanimelist-api","anime_genres_table.csv",pandas_kwargs={"encoding":"ISO-8859-1"})
/tmp/ipykernel_31/1776351084.py:4: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  genresi=kagglehub.load_dataset(kagglehub.KaggleDatasetAdapter.PANDAS,"patmendoza/myanimelist-api","genres_l.csv",pandas_kwargs={"encoding":"ISO-8859-1"})


In [ ]:
tableimdb=pd.read_csv("/kaggle/input/imdb-movies-dataset/imdb_movies.csv")

In [ ]:
arrind=[]
for i,j in tableimdb.fillna("None").iterrows():
  genresp=j["genre"].split(",")
  genresp=[x.strip() for x in genresp]
  arrind.append(genresp)
  #print(type(j["genre"]))



In [ ]:
print(arrind[7])
dtupd=pd.DataFrame({"genre":arrind})
tableimdb.update(dtupd)

['Animation', 'Family', 'Fantasy', 'Adventure', 'Comedy']


In [ ]:
print(tableimdb.head())

                         names       date_x  score  \
0                    Creed III  03/02/2023    73.0   
1     Avatar: The Way of Water  12/15/2022    78.0   
2  The Super Mario Bros. Movie  04/05/2023    76.0   
3                      Mummies  01/05/2023    70.0   
4                    Supercell  03/17/2023    61.0   

                                             genre  \
0                                  [Drama, Action]   
1             [Science Fiction, Adventure, Action]   
2  [Animation, Adventure, Family, Fantasy, Comedy]   
3  [Animation, Comedy, Family, Adventure, Fantasy]   
4                                         [Action]   

                                            overview  \
0  After dominating the boxing world, Adonis Cree...   
1  Set more than a decade after the events of the...   
2  While working underground to fix a water main,...   
3  Through a series of unfortunate events, three ...   
4  Good-hearted teenager William always lived in ...   

             

\

In [ ]:
Genrestableweak=Genrestableweak[Genrestableweak.tm_ky == 2]
maintable=maintable[maintable.tm_ky == 2]
genresi=genresi[genresi.tm_ky == 2]

In [ ]:
joinngenres=Genrestableweak.merge(genresi,on="genres_id").groupby("mal_id").agg({
    'genres_de' : list
})

In [ ]:
print(joinngenres)

In [ ]:
print(maintable.head())

In [ ]:
alltable=maintable.merge(joinngenres,on="mal_id")


In [ ]:
print(alltable.loc[1])

In [ ]:
import random
from transformers import AutoTokenizer, AutoModel

In [ ]:
def psample(dataframe):
  randtn=random.randint(0,len(dataframe)-1)
  genre1=dataframe.loc[randtn]["genre"]
  text1=dataframe.loc[randtn]["overview"]
  if len(text1.split(" ")) < 15:
      return psample(dataframe)

  dataframed=dataframe.loc[dataframe.index > randtn]

  leng=len(genre1)
  #print(leng)
  randomn=random.randint(1,leng)
  genre2=genre1[:randomn]
  #print( genre2)
  genre2=set(genre2)
  for index, i in dataframed.fillna("").iterrows():
    if genre2.issubset(set(i["genre"])) and text1 != i["overview"] and len(i["overview"].split(" "))>15:
      probability =len(genre2)/len(i["genre"])
      #probability=-1+probability*2
      return text1, i["overview"],probability
  print("unsucessful")
  return psample(dataframe)









In [ ]:
psample(tableimdb)

('This is the extraordinary tale of two brothers named Moses and Ramses, one born of royal blood, and one an orphan with a secret past. Growing up the best of friends, they share a strong bond of free-spirited youth and good-natured rivalry. But the truth will ultimately set them at odds, as one becomes the ruler of the most powerful empire on earth, and the other the chosen leader of his people! Their final confrontation will forever change their lives and the world.',
 'Young hobbit Frodo Baggins, after inheriting a mysterious ring from his uncle Bilbo, must leave his home in order to keep it from falling into the hands of its evil creator. Along the way, a fellowship is formed to protect the ringbearer and make sure that the ring arrives at its final destination: Mt. Doom, the only place where it can be destroyed.',
 -0.33333333333333337)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2",model_max_length=100000)
#result=psample(datasett)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
vocab_dict={}
for i,j in tableimdb.fillna("None").iterrows():
  text=j["overview"]
  #print(text)
  tokens=tokenizer.tokenize(text)
  for token in tokens:
    if token not in vocab_dict:
      vocab_dict[token]=len(vocab_dict)


In [ ]:
vocab_dict["<UNK>"]=len(vocab_dict)

In [ ]:
print(vocab_dict["<UNK>"])

18986


In [ ]:
print(datasett.loc[6]["Overview"])

In [ ]:
print(result)


In [ ]:
from einops.layers.torch import Rearrange


In [ ]:
class dowindows(torch.nn.Module):
  def __init__(self,window_size):
    super().__init__()
    self.windowslayer=Rearrange("(b a) w -> a b w",b=window_size)

  def forward(self,x):
    return self.windowslayer(x)

class windowtblock(torch.nn.Module):
  def __init__(self,nheads,embed_dim,bfirst):
    super().__init__()
    self.mha=torch.nn.MultiheadAttention(embed_dim,nheads,batch_first=bfirst,dropout=0.4)
    self.norm1=torch.nn.LayerNorm(embed_dim)
    self.mlp=torch.nn.Sequential(
        torch.nn.Linear(embed_dim,4*embed_dim),
        torch.nn.GELU(),
        torch.nn.Dropout(0.3),
        torch.nn.Linear(4*embed_dim,embed_dim))
    self.norm2=torch.nn.LayerNorm(embed_dim)

  def forward(self,x):
    x=self.norm1(x+self.mha(x,x,x)[0])
    x=self.norm2(x+self.mlp(x))
    return x


class summary(torch.nn.Module):
  def __init__(self,embed_dim,nheads):
    super().__init__()
    parameters=torch.nn.Parameter(torch.randn(1,embed_dim))
    self.parameterss=parameters
    self.mha=torch.nn.MultiheadAttention(embed_dim,nheads,dropout=0.1)
    self.norm=torch.nn.LayerNorm(embed_dim)
    self.mlp=torch.nn.Sequential(
        torch.nn.Linear(embed_dim,4*embed_dim),
        torch.nn.GELU(),
        torch.nn.Dropout(0.3),
        torch.nn.Linear(4*embed_dim,embed_dim))
    self.norm2=torch.nn.LayerNorm(embed_dim)
  def forward(self,x):
    x=self.norm(self.parameterss+self.mha(self.parameterss,x,x)[0])
    x=self.norm2(x+self.mlp(x))
    return x

class PositionalEncoding(torch.nn.Module):

    def __init__(self, d_model: int, dropout: float = 0.3, max_len: int = 5000):
        super().__init__()
        self.dropout = torch.nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-torch.math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Arguments:
            x: Tensor, shape ``[seq_len, batch_size, embedding_dim]``
        """
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)
class transformerr(torch.nn.Module):
  def __init__(self,embed_dim,nheads,bfirst,window_size,vocab_size,vec_dim):
    super().__init__()
    self.embed_dim=embed_dim
    self.nheads=nheads
    self.bfirst=bfirst
    self.window_size=window_size
    self.vocab_size=vocab_size
    self.embedding=torch.nn.Embedding(vocab_size,embed_dim)
    self.window=dowindows(window_size)
    self.cls=torch.nn.Parameter(torch.randn(embed_dim))
    self.fill=torch.nn.Parameter(torch.randn(embed_dim))
    self.pos=PositionalEncoding(embed_dim)
    self.windowtblock=windowtblock(nheads,embed_dim,bfirst)
    self.mhacls=torch.nn.MultiheadAttention(embed_dim,nheads,dropout=0.1)
    self.windowtblock2=windowtblock(nheads,embed_dim,bfirst)
    self.mhacls2=torch.nn.MultiheadAttention(embed_dim,nheads,dropout=0.1)
    self.windowtblock3=windowtblock(nheads,embed_dim,bfirst)
    self.mhacls3=torch.nn.MultiheadAttention(embed_dim,nheads,dropout=0.1)
    self.windowtblock4=windowtblock(nheads,embed_dim,bfirst)
    self.mhacls4=torch.nn.MultiheadAttention(embed_dim,nheads,dropout=0.1)
    self.summary=summary(embed_dim,nheads)
    self.project=torch.nn.Sequential(
        torch.nn.Linear(embed_dim,200),
        torch.nn.GELU(),
        torch.nn.Dropout(0.3),
        torch.nn.Linear(200,100),
        torch.nn.GELU(),
        #torch.nn.Dropout(0.3),
        torch.nn.Linear(100,1)
        )

  def forward(self,x):
    x=self.embedding(x)
    x=x.squeeze(0)
    #print(x.shape)
    while x.shape[0] % self.window_size != 0:
      x=torch.cat((x,self.fill.unsqueeze(0)),dim=0)
    x=self.window(x)

    cls=self.cls.repeat(x.shape[0],1,1)
    #print(x.shape)
    x=torch.cat((cls,x),dim=1)
    #print(x.shape)
    x=x.permute(1,0,2)
    #print(x.shape)
    x=self.pos(x)
    x=x.permute(1,0,2)
    #print(x.shape)
    x=self.windowtblock(x)
    cls=x[:,0,:]
    #print(cls.shape)
    cls=self.mhacls(cls,cls,cls)[0]
    cls=cls.reshape(x.shape[0],1,self.embed_dim)
    x=torch.cat((cls,x[:,1:,:]),dim=1)
    x=self.windowtblock2(x)
    cls=x[:,0,:]
    cls=self.mhacls2(cls,cls,cls)[0]
    cls=cls.reshape(x.shape[0],1,self.embed_dim)
    x=torch.cat((cls,x[:,1:,:]),dim=1)
    x=self.windowtblock3(x)
    cls=x[:,0,:]
    cls=self.mhacls3(cls,cls,cls)[0]
    cls=cls.reshape(x.shape[0],1,self.embed_dim)
    x=torch.cat((cls,x[:,1:,:]),dim=1)
    x=self.windowtblock4(x)
    cls=x[:,0,:]
    cls=self.mhacls4(cls,cls,cls)[0]
    #print(cls.shape)
    #x=torch.cat((cls,x[:,1:,:]),dim=1)
    x=self.summary(cls)
    #print(x.shape)
    x=self.project(x)


    return x






In [ ]:
vocab_dict["[SEP]"]=len(vocab_dict)

In [ ]:
model=transformerr(100,4,True,20,len(vocab_dict),100).to("cuda")


model.load_state_dict(torch.load("moderuproend.pt"))

In [ ]:
print(list(vocab_dict.keys())[0])

after


In [ ]:
model.load_state_dict(torch.load("/kaggle/input/searcher-alpha-alpha-1/pytorch/default/1/moderuproend.pt"))

/tmp/ipykernel_31/3495170826.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/kaggle/input/searcher-alpha-alpha-1/pytorch/default/1/mod

<All keys matched successfully>

In [ ]:
#num parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(num_params)

44457100


In [ ]:
test=torch.randint(0,10,(1,20)).to("cuda")


In [ ]:
#install datasets
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [ ]:
!pip3 install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [ ]:
#download huggingface dataset
#from datasets import load_dataset
#import huggingface datasets

from datasets import load_dataset
dataset=load_dataset("pszemraj/synthetic-text-similarity")
dataset=dataset["train"].to_pandas()

README.md:   0%|          | 0.00/4.02k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00006.parquet:   0%|          | 0.00/289M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00001-of-00006.parquet:   0%|          | 0.00/289M [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00004-of-00006.parquet:   0%|          | 0.00/287M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00005-of-00006.parquet:   0%|          | 0.00/290M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `elproyecto` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `elproye

In [ ]:
sub=dataset.loc[dataset.index<5000]

In [ ]:
vocab_dict={}
for i,j in sub.iterrows():
    firstt=tokenizer.tokenize(j["text1"])
    secondj=tokenizer.tokenize(j["text2"])
    total=firstt+secondj
    for d in total:
        if d not in vocab_dict:
            vocab_dict[d]=len(vocab_dict)
    if i%50==0:
        print(i)




0
50
100
150
200
250
300
350
400
450
500
550
600
650
700
750
800
850
900
950
1000
1050
1100
1150
1200
1250
1300
1350
1400
1450
1500
1550
1600
1650
1700
1750
1800
1850
1900
1950
2000
2050
2100
2150
2200
2250
2300
2350
2400
2450
2500
2550
2600
2650
2700
2750
2800
2850
2900
2950
3000
3050
3100
3150
3200
3250
3300
3350
3400
3450
3500
3550
3600
3650
3700
3750
3800
3850
3900
3950
4000
4050
4100
4150
4200
4250
4300
4350
4400
4450
4500
4550
4600
4650
4700
4750
4800
4850
4900
4950


In [ ]:
model(test).shape


torch.Size([1, 1])

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
#gradient test
#sample=model(test).mean()
#sample.backward()
#print gradients
#for name, param in model.named_parameters():
    #if param.requires_grad:

#print(name, param.grad)
clear_output()

In [ ]:
def to_index(text,updcount,useupdate,minimum):
  #print(text[0])
  text=tokenizer.tokenize(text)
  #print(text)
  settext=set(text)
  filtered=[]
  if useupdate:
      for i in text:
          if i not in updcount or updcount[i]<minimum:
              continue
          filtered.append(i)
      text=filtered
  #print(text)


  #print(text)
  for i in range(len(text)):
    try:
      text[i]=vocab_dict[text[i]]
    except:
      text[i]=vocab_dict["<UNK>"]
      #text[i]=-1

  #print(text)
  #text=[x for x in text if x !=-1]
  return torch.Tensor(text).type(torch.long).unsqueeze(0).to("cuda"),settext

In [ ]:
def cosine_regression_loss(x1, x2, target_similarity,margin):
    actual_sim = torch.functional.F.cosine_similarity(x1, x2)
    target_sim_with_margin = torch.where(
        target_similarity > 0.5,
        torch.clamp(2 * target_similarity - 1 + margin, min=-1.0, max=1.0),  # Similar pairs
        torch.clamp(2 * target_similarity - 1 - margin, min=-1.0, max=1.0)  # Dissimilar pairs
    )
    return torch.functional.F.mse_loss(actual_sim, target_sim_with_margin)
from IPython.display import clear_output

In [ ]:
to_index(sub.loc[3000]["text1"])




TypeError: to_index() missing 3 required positional arguments: 'updcount', 'useupdate', and 'minimum'

In [ ]:
import torch

class PyTorchMinMaxScaler:
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range
        self.data_min = None
        self.data_max = None
        self.data_range = None
        self.scale_ = None
        self.min_ = None

    def fit(self, data):
        self.data_min = torch.min(data, dim=0, keepdim=True)[0]
        self.data_max = torch.max(data, dim=0, keepdim=True)[0]
        self.data_range = self.data_max - self.data_min
        # Prevent division by zero
        self.data_range[self.data_range == 0] = 1
        self.scale_ = (self.feature_range[1] - self.feature_range[0]) / self.data_range
        self.min_ = self.feature_range[0] - self.data_min * self.scale_

    def transform(self, data):
        return (data - self.data_min) * self.scale_ + self.feature_range[0]  # or self.min_

    def inverse_transform(self, data):
        return (data - self.feature_range[0]) / self.scale_ + self.data_min

In [ ]:
import random

In [ ]:

scaler=PyTorchMinMaxScaler()
tensors=[]
with torch.no_grad():
 for i in range(0,200):
   randn=random.randint(0,len(sub)-1)
   text1=sub.loc[randn]["text1"]
   text2=sub.loc[randn]["text2"]
   textfinal=f"{text1}[SEP]{text2}"
   textfinal,_=to_index(textfinal,None,False,None)
   pred=model(textfinal)
   tensors.append(pred)

#matrix
 scaler.fit(torch.stack(tensors))




In [ ]:

tensors=[]
with torch.no_grad():
 for i in range(0,200):
   randn=random.randint(0,len(sub)-1)
   text1=sub.loc[randn]["text1"]
   text2=sub.loc[randn]["text2"]
   textfinal=f"{text1}[SEP]{text2}"
   textfinal,_=to_index(textfinal,None,False,None)
   pred=model(textfinal)
   tensors.append(pred)

#matrix
 #scaler.fit(torch.stack(tensors))
 matrix=scaler.transform(torch.stack(tensors))




In [ ]:
print(matrix)

tensor([[[0.5124]],

        [[0.2134]],

        [[0.3358]],

        [[0.2686]],

        [[0.6117]],

        [[0.5017]],

        [[0.2795]],

        [[0.0271]],

        [[0.5521]],

        [[0.2260]],

        [[0.3867]],

        [[0.6706]],

        [[0.7863]],

        [[0.2148]],

        [[0.4183]],

        [[0.3042]],

        [[0.6177]],

        [[0.2357]],

        [[0.5367]],

        [[0.8400]],

        [[0.3576]],

        [[0.2948]],

        [[0.4221]],

        [[0.0896]],

        [[0.6380]],

        [[0.6437]],

        [[0.7186]],

        [[0.4384]],

        [[0.3554]],

        [[0.7094]],

        [[0.4106]],

        [[0.4308]],

        [[0.3969]],

        [[0.0778]],

        [[0.4837]],

        [[0.4914]],

        [[0.7568]],

        [[0.2297]],

        [[0.3368]],

        [[0.2309]],

        [[0.2345]],

        [[0.4000]],

        [[0.5709]],

        [[0.1412]],

        [[0.0780]],

        [[0.7085]],

        [[0.3342]],

        [[0.5

In [ ]:
def continousEmbLoss(emb1,emb2,target_sim):
  binaryloss=torch.nn.CosineEmbeddingLoss(margin=0.5)
  binarysim=1 if target_sim >= 0.5 else -1
  binarysim=torch.Tensor([binarysim]).to("cuda")
  #target_sim=target_sim if binarysim == 1 else 1-target_sim
  return binaryloss(emb1,emb2,binarysim)


In [ ]:
#training loop
model.train()
epochs=10000
learning_rate=0.00002
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)
#loss_fn=cosine_regression_loss
loss_fn=torch.nn.L1Loss()

optimizer.zero_grad()
lossvector=[]

acc=10
#alltable=alltable.fillna("None")
updatedict={}
setupdate=set([])
for i in range(epochs):
  number=random.randint(0,4999)
  #text1,text2,probability=psample(tableimdb)
  inst=sub.loc[number]
  text1=inst["text1"]
  text2=inst["text2"]
  #probability=inst["label"]*2 -1
  probability=inst["label"]
  finaltxt=f"{text1}[SEP]{text2}"
  #text1,set1=to_index(text1,"_",False,"_")
  #text2,set2=to_index(text2,"_",False,"_")
  finaltxt,sett=to_index(finaltxt,None,False,None)


  setupdate=setupdate.union(sett)
  #print(text1.shape)
  #print(text2.shape)
  #text1=torch.Tensor(text1).type(torch.long).unsqueeze(0).to("cuda")
  #text2=torch.Tensor(text2).type(torch.long).unsqueeze(0).to("cuda")
  #output1=model(text1)
  #output2=model(text2)
  output=model(finaltxt)
  loss=loss_fn(output,torch.Tensor([probability])).to("cuda")/acc

  #output1=output[0]
  #output2=output[1]
  #print(output1.shape)
  #print(output2.shape)
  #loss=loss_fn(output1,output2,torch.Tensor([probability]).to("cuda"),0.5)/acc
  #loss=continousEmbLoss(output1,output2,probability)
  lossvector.append(loss.item())
  loss.backward()
  print(output)
  if (i+1)%acc==0:
    torch.nn.utils.clip_grad_norm(model.parameters(),1.0)
    optimizer.step()
    for j in list(setupdate):
        try:
            updatedict[j]=updatedict[j]+1
        except:
            updatedict[j]=1
    setupdate=set([])

    #save gradients
    with open("grad.txt","w") as f:
      for name, param in model.named_parameters():
        if param.requires_grad:
          f.write(name+"\n")
          f.write(str(param.grad)+"\n")

    optimizer.zero_grad()
    plt.plot(lossvector)
    plt.show()
    plt.close()
    #cosine similarity
    #print(torch.cosine_similarity(output1,output2))
    print(output)
    print(probability)
  if (i+1)%100==0:
      clear_output()





RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
print(len(updatedict))

18097


In [ ]:
import json

In [ ]:
#test
model.eval()
text1=""" Ever since the death of his father, the burden of supporting the family has fallen upon Tanjirou Kamado's shoulders. Though living impoverished on a remote mountain, the Kamado family are able to enjoy a relatively peaceful and happy life. One day, Tanjirou decides to go down to the local village to make a little money selling charcoal. On his way back, night falls, forcing Tanjirou to take shelter in the house of a strange man, who warns him of the existence of flesh-eating demons that lurk in the woods at night.

#When he finally arrives back home the next day, he is met with a horrifying sight—his whole family has been slaughtered. Worse still, the sole survivor is his sister Nezuko, who has been turned into a bloodthirsty demon. Consumed by rage and hatred, Tanjirou swears to avenge his family and stay by his only remaining sibling. Alongside the mysterious group calling themselves the Demon Slayer Corps, Tanjirou will do whatever it takes to slay the demons and protect the remnants of his beloved sister's humanity.
#"""
#text2="primarily focuses on the daily antics of a trio of childhood friends—high school girls , and  stories soon intertwine with the young genius, her robot caretaker, and their talking cat. With every passing day, the lives of these six, as well as of the many people around them, experience both the calms of normal life and the insanity of the absurd. Walking to school, being bitten by a talking crow, spending time with friends, and watching the principal suplex a deer: they are all in a day's work in the extraordinary everyday lives of those."
#text1="""Centuries ago, mankind was slaughtered to near extinction by monstrous humanoid creatures called Titans, forcing humans to hide in fear behind enormous concentric walls. What makes these giants truly terrifying is that their taste for human flesh is not born out of hunger but what appears to be out of pleasure. To ensure their survival, the remnants of humanity began living within defensive barriers, resulting in one hundred years without a single titan encounter. However, that fragile calm is soon shattered when a colossal Titan manages to breach the supposedly impregnable outer wall, reigniting the fight for survival against the man-eating abominations.


#After witnessing a horrific personal loss at the hands of the invading creatures, dedicates his life to their eradication by enlisting into the Survey Corps, an elite military unit that combats the merciless humanoids outside the protection of the walls.  his adopted sister , and his childhood friend  join the brutal war against the Titans and race to discover a way of defeating them before the last walls are breached."""
text2="its about a kid that sells charcoal and lose his family  and only his sister  survives"
#text2=" with a lot of features"
#text1=tokenizer.tokenize(text1)
#print(text2)
#text2=tokenizer.tokenize(text2)
text1,_=to_index(text1,updatedict,True,10)
text2,_=to_index(text2,updatedict,True,10)

#print(text2)

#text1=text1.unsqueeze(0)
#text2=text2.unsqueeze(0)
output1=model(text1)
output2=model(text2)
print(torch.cosine_similarity(output1,output2))

tensor([0.8799], device='cuda:0', grad_fn=<SumBackward1>)


In [ ]:
tableimdb.set_index(pd.Index(list(range(len(tableimdb)))),inplace=True)


In [ ]:
print(0)

0


In [ ]:
print(tableimdb.loc[0])

names                                                 Creed III
date_x                                              03/02/2023 
score                                                      73.0
genre                                           [Drama, Action]
overview      After dominating the boxing world, Adonis Cree...
crew          Michael B. Jordan, Adonis Creed, Tessa Thompso...
orig_title                                            Creed III
status                                                 Released
orig_lang                                               English
budget_x                                             75000000.0
revenue                                             271616668.0
country                                                      AU
Name: 0, dtype: object


In [ ]:
model.eval()

transformerr(
  (embedding): Embedding(28311, 600)
  (window): dowindows(
    (windowslayer): Rearrange('(b a) w -> a b w', b=20)
  )
  (pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (windowtblock): windowtblock(
    (mha): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=600, out_features=600, bias=True)
    )
    (norm1): LayerNorm((600,), eps=1e-05, elementwise_affine=True)
    (mlp): Sequential(
      (0): Linear(in_features=600, out_features=2400, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=2400, out_features=600, bias=True)
    )
    (norm2): LayerNorm((600,), eps=1e-05, elementwise_affine=True)
  )
  (mhacls): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=600, out_features=600, bias=True)
  )
  (windowtblock2): windowtblock(
    (mha): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=600, out_features=600, bias=True)


model.eval()

In [ ]:
#save model
torch.save(model.state_dict(),"moderuproend.pt")

In [ ]:
import json

with open('datadictupd.json', 'w') as fp:
    json.dump(updatedict, fp)

In [ ]:
#vectorize text
vectdict={}
avg=sum(updatedict.values())/len(updatedict)
with torch.no_grad():
 for i,j in tableimdb.iterrows():
     text=j["overview"]

     if len(text.split(" ")) <15:
         continue
     text,_=to_index(text,updatedict,True,10)
     text=model(text)
     vectdict[text]=i


In [ ]:
updatedict=json.load(open("/kaggle/input/updtddd/datadictupd.json"))

In [ ]:
print(avg)

148.25211187619792


In [ ]:
print(list(vectdict.keys())[1])

tensor([[ 0.0228,  0.0056,  0.5975, -0.1783, -0.1424, -0.1099,  0.0443, -0.4719,
         -0.3729,  0.0672,  0.3556,  0.3764,  0.1956,  0.2976,  0.3139,  0.3043,
          0.1929, -0.2368,  0.3561, -0.4258,  0.2366, -0.3455,  0.0635, -0.3498,
          0.6368,  0.1710,  0.3203, -0.2638,  0.4659,  0.1196, -0.3218,  0.1480,
         -0.3288, -0.5483, -0.5551, -0.3404,  0.1454,  0.0364,  0.2301, -0.1605,
         -0.3736,  0.0342, -0.3389, -0.3043,  0.3334, -0.3743, -0.0711,  0.2929,
          0.0030,  0.2936,  0.2031,  0.5328, -0.3450, -0.7351,  0.1610, -0.3202,
         -0.1696, -0.0407, -0.3209, -0.2976, -0.1240, -0.3184, -0.1628, -0.1378,
          0.1394,  0.0771,  0.0877,  0.4157,  0.3800, -0.4048,  0.1155, -0.0232,
         -0.2166,  0.1773, -0.6113, -0.1993, -0.0174, -0.0769, -0.4497, -0.2866,
         -0.2226,  0.4219, -0.0045,  0.0120, -0.4926, -0.1629, -0.3198,  0.2589,
         -0.7870,  0.0169, -0.1522,  0.2243, -0.2276,  0.2929, -0.0479,  0.2914,
          0.1563,  0.7231, -

In [ ]:
textinp="the movie its about a pet livng his life"
print(avg)
textinp,_=to_index(textinp,updatedict,True,10)
print(textinp)
textinp=model(textinp)
searchdict={}

for i in vectdict:
    searchdict[torch.cosine_similarity(textinp,i)]=vectdict[i]



33.62199691426053
tensor([[   2,  318,  263,  691,   21, 8252, 3230, 1294,   14,   18]],
       device='cuda:0')


In [ ]:
model.eval()

transformerr(
  (embedding): Embedding(28311, 600)
  (window): dowindows(
    (windowslayer): Rearrange('(b a) w -> a b w', b=20)
  )
  (pos): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (windowtblock): windowtblock(
    (mha): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=600, out_features=600, bias=True)
    )
    (norm1): LayerNorm((600,), eps=1e-05, elementwise_affine=True)
    (mlp): Sequential(
      (0): Linear(in_features=600, out_features=2400, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=2400, out_features=600, bias=True)
    )
    (norm2): LayerNorm((600,), eps=1e-05, elementwise_affine=True)
  )
  (mhacls): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=600, out_features=600, bias=True)
  )
  (windowtblock2): windowtblock(
    (mha): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=600, out_features=600, bias=True)


In [ ]:
result=list(searchdict.keys())
result.sort()
idx=searchdict[result[-5]]
print(result[-10])
xd=tableimdb.loc[idx]["overview"]
print(xd)


tensor([0.9959], device='cuda:0', grad_fn=<SumBackward1>)
After witnessing a haunting in their hospital, two doctors become dangerously obsessed with obtaining scientific proof that ghosts exist.


#vectorize

#vectorize text
for